# Financial Q&A 10-K — LLM Financial Question Answering:

This notebook builds a lightweight LLM-based question-answering pipeline using 50 samples from the Financial Q&A 10-K dataset.

The pipeline will:

1. Load and validate the dataset
2. Select exactly 50 valid samples
3. Generate answers using an LLM
4. Compare generated answers with reference answers
5. Evaluate and analyze the results

## 1. Setup:

In [1]:
import sys 
import pandas as pd 

print("Python:", sys.executable)
print("Pandas:", pd.__version__)

Python: /Users/yagyansh/Desktop/pwc-take-home-task/.venv/bin/python
Pandas: 3.0.5


## 2. Loading Dataset:

In [2]:
RAW_DATA_PATH = "/Users/yagyansh/Desktop/pwc-take-home-task/data/Financial-QA-10k.csv"

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

Shape: (7000, 5)
Columns: ['question', 'answer', 'context', 'ticker', 'filing']


## 3. Create the 50-Sample Evaluation Set:

I am supposed to be using only 50 samples which is why I am removing rows with missing values in the fields required for question answering.

In [3]:
raw_df.head(5)

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [4]:
required_columns = ["question", "context", "answer"]
valid_df = raw_df.dropna(subset=required_columns).copy()

print("Valid Rows:", len(valid_df))

Valid Rows: 6997


In [5]:
samples_50 = valid_df.head(50).copy()
print("Samples Shape:", samples_50.shape)

Samples Shape: (50, 5)


In [6]:
samples_50.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [7]:
OUTPUT_PATH = "/Users/yagyansh/Desktop/pwc-take-home-task/data/samples_50.csv"

samples_50.to_csv(OUTPUT_PATH, index = False)
print(f"Saved samples to {OUTPUT_PATH}")

Saved samples to /Users/yagyansh/Desktop/pwc-take-home-task/data/samples_50.csv


In [8]:
check_df = pd.read_csv(OUTPUT_PATH)

print("Verified shapeL", check_df.shape)
print("Columns:", check_df.columns.tolist())

Verified shapeL (50, 5)
Columns: ['question', 'answer', 'context', 'ticker', 'filing']


## 4. Prompt Construction

The model receives the question and the supporting context. The reference answer **("answer" column)** is excluded from the prompt because it is used as ground truth for evaluation.


In [9]:
def build_prompt(question, context):
    return f"""
You are a financial question-answering assistant.

Answer the question using ONLY the information contained in the provided context.

Rules:
- Do not use external knowledge or information from your training data.
- Do not make assumptions that are not supported by the context.
- Do not invent facts or figures.
- If the context does not contain enough information to answer the question reliably,
  abstain by returning "ABSTAIN".
- Keep the answer concise and directly answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

In [10]:
test_sample = samples_50.iloc[0]

prompt = build_prompt(
    test_sample["question"],
    test_sample["context"]
)

print(prompt)


You are a financial question-answering assistant.

Answer the question using ONLY the information contained in the provided context.

Rules:
- Do not use external knowledge or information from your training data.
- Do not make assumptions that are not supported by the context.
- Do not invent facts or figures.
- If the context does not contain enough information to answer the question reliably,
  abstain by returning "ABSTAIN".
- Keep the answer concise and directly answer the question.

Context:
Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

Question:
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

Answer:



In [11]:
print("Question Length:", len(test_sample["question"]))
print("Context Length:", len(test_sample["context"]))
print("Reference Answer Length:", len(test_sample["answer"]))

Question Length: 99
Context Length: 128
Reference Answer Length: 40


In [12]:
print("Reference answer:")
print(test_sample["answer"])

Reference answer:
NVIDIA initially focused on PC graphics.


## 5. Model Selection:

Choosing 2 different LLM models from the OpenAI family of models especially given their recent dominance against Anthropic. Both these models will be evaluated using the same 50 examples and the same prompt structure.


- Model A: GPT-5.6 Sol (`gpt-5.6-sol`) — selected as the quality/reliability-oriented model
- Model B: GPT-5.6 Luna (`gpt-5.6-luna`) — selected as the cost-oriented model

Both Models A & B will be compared on:
- Answer correctness
- Context adherence
- Abstention behaviour
- Consistency
- Cost

The models will receive only the question and supporting context from the dataset to support them in answering the questions asked by our users.


In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

print("API key loaded successfully.")

API key loaded successfully.


In [14]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully.")

OpenAI client created successfully.


In [15]:
models = client.models.list()

for model in models.data:
    if any(name in model.id.lower() for name in ["gpt-5", "gpt-4.1", "gpt-4o"]):
        print(model.id)


gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
gpt-4o-mini-tts
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-5-codex
gpt-5-pro-2025-10-06
gpt-5-pro
gpt-5-search-api
gpt-5-search-api-2025-10-14
gpt-5.1-chat-latest
gpt-5.1-2025-11-13
gpt-5.1
gpt-5.1-codex
gpt-5.1-codex-mini
gpt-5.1-codex-max
gpt-5.2-2025-12-11
gpt-5.2
gpt-5.2-pro-2025-12-11
gpt-5.2-pro
gpt-5.2-chat-latest
gpt-4o-mini-transcribe-2025-12-15
gpt-4o-mini-transcribe-2025-03-20
gpt-4o-mini-tts-2025-03-20
gpt-4o-mini-tts-2025-12-15
gpt-5.2-codex
gpt-5.3-codex
gpt-4o-search-preview
gpt-4o-search-preview-2025-03-11
gpt-5.3-chat-latest
gpt-5.4-2026-03-05
gpt-5.4-pro
gpt-5.4-pro-2026-03-05
g

In [16]:
MODEL_A = "gpt-5.6-sol"
MODEL_B = "gpt-5.6-luna"

print("Model A:", MODEL_A)
print("Model B:", MODEL_B)

Model A: gpt-5.6-sol
Model B: gpt-5.6-luna


## 6. API Call Model Test Run:


In [17]:
test_sample = samples_50.iloc[0]

test_prompt = build_prompt(
    test_sample["question"],
    test_sample["context"]
)

print("QUESTION:")
print(test_sample["question"])

print("\nREFERENCE ANSWER:")
print(test_sample["answer"])

QUESTION:
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

REFERENCE ANSWER:
NVIDIA initially focused on PC graphics.


## Model A:

In [ ]:
response = client.responses.create(
    model=MODEL_A,
    input=test_prompt,
    reasoning={"effort": "none"} # mentioned reasoning equal to none because we are doing straightforward context-based extraction
)

model_a_answer = response.output_text

print("\nMODEL A ANSWER:")
print(model_a_answer)


MODEL A ANSWER:
PC graphics.


In [19]:
response = client.responses.create(
    model=MODEL_B,
    input=test_prompt,
    reasoning={"effort": "none"} # mentioned reasoning equal to none because we are doing straightforward context-based extraction
)

model_b_answer = response.output_text

print("\nMODEL B ANSWER:")
print(model_b_answer)


MODEL B ANSWER:
PC graphics


## 7. Generating Predictions:

In [20]:
def generate_answer(model_name, question, context):
    prompt = build_prompt(question, context)

    response = client.responses.create(
        model = model_name, 
        input = prompt,
        reasoning = {"effort": "none"}
    )

    answer = response.output_text.strip()

    return {
        "predicted_answer": answer,
        "abstained": answer.upper() == "ABSTAIN",
        "response_id": response.id
    }

In [21]:
test_result_a = generate_answer(
    MODEL_A, 
    test_sample["question"],
    test_sample["context"]
)

test_result_b = generate_answer(
    MODEL_B,
    test_sample["question"],
    test_sample["context"]
)

print("MODEL A:")
print(test_result_a)

print("MODEL B:")
print(test_result_b)

MODEL A:
{'predicted_answer': 'PC graphics.', 'abstained': False, 'response_id': 'resp_0f500545e32ba2ae006a9c880d3fbc87d2a59bfcfaf03586e5'}
MODEL B:
{'predicted_answer': 'PC graphics', 'abstained': False, 'response_id': 'resp_0b2603a895c70825006a9c880ed2a887d28509d10cdd1987ab'}


In [22]:
import time

def run_model_on_samples(model_name, samples):
    results = []

    for idx, row in samples.iterrows():
        result = generate_answer(
            model_name,
            row["question"],
            row["context"]
        )

        results.append({
            "sample_id": idx + 1,
            "question": row["question"],
            "context": row["context"],
            "reference_answer": row["answer"],
            "predicted_answer": result["predicted_answer"],
            "abstained": result["abstained"],
            "response_id": result["response_id"],
            "model": model_name
        })

        print(f"{model_name}: completed {len(results)}/{len(samples)}")

    return pd.DataFrame(results)

In [23]:
results_a = run_model_on_samples(
    MODEL_A,
    samples_50
)


gpt-5.6-sol: completed 1/50
gpt-5.6-sol: completed 2/50
gpt-5.6-sol: completed 3/50
gpt-5.6-sol: completed 4/50
gpt-5.6-sol: completed 5/50
gpt-5.6-sol: completed 6/50
gpt-5.6-sol: completed 7/50
gpt-5.6-sol: completed 8/50
gpt-5.6-sol: completed 9/50
gpt-5.6-sol: completed 10/50
gpt-5.6-sol: completed 11/50
gpt-5.6-sol: completed 12/50
gpt-5.6-sol: completed 13/50
gpt-5.6-sol: completed 14/50
gpt-5.6-sol: completed 15/50
gpt-5.6-sol: completed 16/50
gpt-5.6-sol: completed 17/50
gpt-5.6-sol: completed 18/50
gpt-5.6-sol: completed 19/50
gpt-5.6-sol: completed 20/50
gpt-5.6-sol: completed 21/50
gpt-5.6-sol: completed 22/50
gpt-5.6-sol: completed 23/50
gpt-5.6-sol: completed 24/50
gpt-5.6-sol: completed 25/50
gpt-5.6-sol: completed 26/50
gpt-5.6-sol: completed 27/50
gpt-5.6-sol: completed 28/50
gpt-5.6-sol: completed 29/50
gpt-5.6-sol: completed 30/50
gpt-5.6-sol: completed 31/50
gpt-5.6-sol: completed 32/50
gpt-5.6-sol: completed 33/50
gpt-5.6-sol: completed 34/50
gpt-5.6-sol: completed 

In [25]:
results_b = run_model_on_samples(
    MODEL_B,
    samples_50
)

gpt-5.6-luna: completed 1/50
gpt-5.6-luna: completed 2/50
gpt-5.6-luna: completed 3/50
gpt-5.6-luna: completed 4/50
gpt-5.6-luna: completed 5/50
gpt-5.6-luna: completed 6/50
gpt-5.6-luna: completed 7/50
gpt-5.6-luna: completed 8/50
gpt-5.6-luna: completed 9/50
gpt-5.6-luna: completed 10/50
gpt-5.6-luna: completed 11/50
gpt-5.6-luna: completed 12/50
gpt-5.6-luna: completed 13/50
gpt-5.6-luna: completed 14/50
gpt-5.6-luna: completed 15/50
gpt-5.6-luna: completed 16/50
gpt-5.6-luna: completed 17/50
gpt-5.6-luna: completed 18/50
gpt-5.6-luna: completed 19/50
gpt-5.6-luna: completed 20/50
gpt-5.6-luna: completed 21/50
gpt-5.6-luna: completed 22/50
gpt-5.6-luna: completed 23/50
gpt-5.6-luna: completed 24/50
gpt-5.6-luna: completed 25/50
gpt-5.6-luna: completed 26/50
gpt-5.6-luna: completed 27/50
gpt-5.6-luna: completed 28/50
gpt-5.6-luna: completed 29/50
gpt-5.6-luna: completed 30/50
gpt-5.6-luna: completed 31/50
gpt-5.6-luna: completed 32/50
gpt-5.6-luna: completed 33/50
gpt-5.6-luna: compl

In [26]:
results_a[[
    "sample_id",
    "reference_answer",
    "predicted_answer",
    "abstained"
]].head(10)

,sample_id,reference_answer,predicted_answer,abstained
0,1,NVIDIA initially focused on PC graphics.,PC graphics.,False
1,2,Recent applications of GPU-powered deep learni...,Recent applications include recommendation sys...,False
2,3,NVIDIA invented the GPU in 1999.,NVIDIA invented the GPU in 1999.,False
3,4,NVIDIA's platform strategy brings together har...,"It integrates hardware, systems, software, alg...",False
4,5,NVIDIA's CUDA programming model opened the par...,It enables the GPU’s parallel processing capab...,False
5,6,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",False
6,7,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement due to significa...,False
7,8,NVIDIA recorded an acquisition termination cos...,$1.35 billion.,False
8,9,The NVIDIA computing platform focuses on accel...,"The most compute-intensive workloads, includin...",False
9,10,The NVIDIA computing platform includes energy-...,The NVIDIA computing platform includes energy-...,False


In [27]:
results_a

,sample_id,question,context,reference_answer,predicted_answer,abstained,response_id,model
0,1,What area did NVIDIA initially focus on before...,"Since our original focus on PC graphics, we ha...",NVIDIA initially focused on PC graphics.,PC graphics.,False,resp_064193139907dedf006a9c884847d087d2a3d3573...,gpt-5.6-sol
1,2,What are some of the recent applications of GP...,Some of the most recent applications of GPU-po...,Recent applications of GPU-powered deep learni...,Recent applications include recommendation sys...,False,resp_0e5530428dab770f006a9c884a547c87d2bdea6c0...,gpt-5.6-sol
2,3,What significant invention did NVIDIA create i...,Our invention of the GPU in 1999 defined moder...,NVIDIA invented the GPU in 1999.,NVIDIA invented the GPU in 1999.,False,resp_0186743ca973334f006a9c884c33e087d2bf69a11...,gpt-5.6-sol
3,4,How does NVIDIA's platform strategy contribute...,"NVIDIA has a platform strategy, bringing toget...",NVIDIA's platform strategy brings together har...,"It integrates hardware, systems, software, alg...",False,resp_0bd634d092f2c27b006a9c884dbb0c87d29904dbc...,gpt-5.6-sol
4,5,What does NVIDIA's CUDA programming model enable?,With our introduction of the CUDA programming ...,NVIDIA's CUDA programming model opened the par...,It enables the GPU’s parallel processing capab...,False,resp_081c4e4462fc97ab006a9c884f1cd087d2bae2b02...,gpt-5.6-sol
5,6,What industries use NVIDIA's GPUs and software...,A rapidly growing number of enterprises and st...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",False,resp_080dc67e12527bbe006a9c88508eac87d2ac6de1e...,gpt-5.6-sol
6,7,Why did NVIDIA and SoftBank terminate their Sh...,Termination of the Arm Share Purchase Agreemen...,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement due to significa...,False,resp_07fb1090b38ade9b006a9c8851c50887d29316321...,gpt-5.6-sol
7,8,What amount did NVIDIA record as an acquisitio...,We recorded an acquisition termination cost of...,NVIDIA recorded an acquisition termination cos...,$1.35 billion.,False,resp_0b6f12fa8e11d6f2006a9c885401f487d2a92e007...,gpt-5.6-sol
8,9,What does the NVIDIA computing platform focus ...,Data Center The NVIDIA computing platform is f...,The NVIDIA computing platform focuses on accel...,"The most compute-intensive workloads, includin...",False,resp_0de97e45ccc11fba006a9c88561bf487d2bababf2...,gpt-5.6-sol
9,10,What are the key components of the NVIDIA comp...,Data Center The NVIDIA computing platform is f...,The NVIDIA computing platform includes energy-...,The NVIDIA computing platform includes energy-...,False,resp_0fc031d32f494816006a9c8857567887d2b85d039...,gpt-5.6-sol


In [29]:
results_b

,sample_id,question,context,reference_answer,predicted_answer,abstained,response_id,model
0,1,What area did NVIDIA initially focus on before...,"Since our original focus on PC graphics, we ha...",NVIDIA initially focused on PC graphics.,PC graphics,False,resp_0c7db54a1506b55d006a9c88a7dcb887d28116ca5...,gpt-5.6-luna
1,2,What are some of the recent applications of GP...,Some of the most recent applications of GPU-po...,Recent applications of GPU-powered deep learni...,"Recommendation systems, large language models,...",False,resp_0039eb8509214da7006a9c88a9190c87d2998aa29...,gpt-5.6-luna
2,3,What significant invention did NVIDIA create i...,Our invention of the GPU in 1999 defined moder...,NVIDIA invented the GPU in 1999.,The GPU (graphics processing unit).,False,resp_02a02537b7264faf006a9c88aa5d9487d2a2356d6...,gpt-5.6-luna
3,4,How does NVIDIA's platform strategy contribute...,"NVIDIA has a platform strategy, bringing toget...",NVIDIA's platform strategy brings together har...,"It brings together hardware, systems, software...",False,resp_01208a92bd3eaf10006a9c88ab474887d28b2579f...,gpt-5.6-luna
4,5,What does NVIDIA's CUDA programming model enable?,With our introduction of the CUDA programming ...,NVIDIA's CUDA programming model opened the par...,It enables general-purpose computing using the...,False,resp_09799a7913e30515006a9c88ac64b087d2a53a23b...,gpt-5.6-luna
5,6,What industries use NVIDIA's GPUs and software...,A rapidly growing number of enterprises and st...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",False,resp_0bb6abd7c82017c2006a9c88adb01887d29dc0640...,gpt-5.6-luna
6,7,Why did NVIDIA and SoftBank terminate their Sh...,Termination of the Arm Share Purchase Agreemen...,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement because signific...,False,resp_037a52475abbfa4b006a9c88aea67887d2a708507...,gpt-5.6-luna
7,8,What amount did NVIDIA record as an acquisitio...,We recorded an acquisition termination cost of...,NVIDIA recorded an acquisition termination cos...,$1.35 billion.,False,resp_09bd273e5cd0eeaf006a9c88afe4f887d2aa3836f...,gpt-5.6-luna
8,9,What does the NVIDIA computing platform focus ...,Data Center The NVIDIA computing platform is f...,The NVIDIA computing platform focuses on accel...,The NVIDIA computing platform focuses on accel...,False,resp_0f8809f0396c00cd006a9c88b14c3c87d2b688183...,gpt-5.6-luna
9,10,What are the key components of the NVIDIA comp...,Data Center The NVIDIA computing platform is f...,The NVIDIA computing platform includes energy-...,"Energy-efficient GPUs, data processing units (...",False,resp_02fa5b427a428223006a9c88b26ca887d29811be9...,gpt-5.6-luna


## 8. Evaluation:

In [30]:
import re

def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s.%$-]", "", text)

    return text

In [31]:
def exact_match(predicted, reference):
    return normalize_text(predicted) == normalize_text(reference)

In [32]:
results_a["exact_match"] = results_a.apply(
    lambda row: exact_match(
        row["predicted_answer"],
        row["reference_answer"]
    ),
    axis=1
)

results_b["exact_match"] = results_b.apply(
    lambda row: exact_match(
        row["predicted_answer"],
        row["reference_answer"]
    ),
    axis=1
)


In [33]:
print("Model A exact matches:", results_a["exact_match"].sum())
print("Model B exact matches:", results_b["exact_match"].sum())

Model A exact matches: 4
Model B exact matches: 3


## 9. LLM-AS-A-Judge:

In [34]:
def build_evaluation_prompt(question, context, reference_answer, predicted_answer):
    return f"""

You are evaluating a financial question-answering system.

Your task is to determine whether the predicted answer is correct based ONLY
on the provided question, context, and reference answer.

Evaluate the predicted answer using these rules:

- CORRECT: The prediction conveys the same substantive answer as the reference
  and is supported by the context.
- INCORRECT: The prediction contains a materially wrong answer, unsupported
  information, or contradicts the reference/context.
- ABSTAINED: The prediction explicitly says ABSTAIN or indicates that there
  is insufficient information.

Do not use external knowledge or any form of tools like MCP or RAG.

Question:
{question}

Context:
{context}

Reference answer:
{reference_answer}

Predicted answer:
{predicted_answer}

Return exactly one label:
CORRECT
INCORRECT
ABSTAINED
"""


In [39]:
JUDGE_MODEL = "gpt-4o-mini"

In [41]:
evaluation_sample = results_a.iloc[0]

evaluation_prompt = build_evaluation_prompt(
    evaluation_sample["question"],
    evaluation_sample["context"],
    evaluation_sample["reference_answer"],
    evaluation_sample["predicted_answer"]
)

judge_response = client.responses.create(
    model=JUDGE_MODEL,
    input=evaluation_prompt
)

judge_result = judge_response.output_text.strip()

print("Judge result:", judge_result)

Judge result: CORRECT


## 9. Semantic Evaluation:

In [42]:
def evaluate_prediction(
    question,
    context,
    reference_answer,
    predicted_answer
):
    evaluation_prompt = build_evaluation_prompt(
        question,
        context,
        reference_answer,
        predicted_answer
    )

    response = client.responses.create(
        model=JUDGE_MODEL,
        input=evaluation_prompt
    )

    result = response.output_text.strip().upper()

    # Keep only valid labels
    if result not in {"CORRECT", "INCORRECT", "ABSTAINED"}:
        result = "UNKNOWN"

    return result

In [44]:
test_eval_a = evaluate_prediction(
    results_a.iloc[0]["question"],
    results_a.iloc[0]["context"],
    results_a.iloc[0]["reference_answer"],
    results_a.iloc[0]["predicted_answer"]
)

print("Model A evaluation:", test_eval_a)

Model A evaluation: CORRECT


In [45]:
results_a["evaluation"] = results_a.apply(
    lambda row: evaluate_prediction(
        row["question"],
        row["context"],
        row["reference_answer"],
        row["predicted_answer"]
    ),
    axis=1
)

In [46]:
results_a["evaluation"].value_counts()

evaluation
CORRECT      49
INCORRECT     1
Name: count, dtype: int64

In [47]:
results_b["evaluation"] = results_b.apply(
    lambda row: evaluate_prediction(
        row["question"],
        row["context"],
        row["reference_answer"],
        row["predicted_answer"]
    ),
    axis=1
)


In [48]:
results_b["evaluation"].value_counts()

evaluation
CORRECT      49
INCORRECT     1
Name: count, dtype: int64

In [49]:
def calculate_metrics(results):
    total = len(results)

    correct = (results["evaluation"] == "CORRECT").sum()
    incorrect = (results["evaluation"] == "INCORRECT").sum()
    abstained = (results["evaluation"] == "ABSTAINED").sum()

    answered = total - abstained

    accuracy = correct / total
    answer_accuracy = correct / answered if answered > 0 else 0
    abstention_rate = abstained / total
    coverage = answered / total

    return {
        "total": total,
        "correct": correct,
        "incorrect": incorrect,
        "abstained": abstained,
        "accuracy": accuracy,
        "answer_accuracy": answer_accuracy,
        "abstention_rate": abstention_rate,
        "coverage": coverage
    }


In [50]:
metrics_a = calculate_metrics(results_a)
metrics_b = calculate_metrics(results_b)

metrics_a, metrics_b

({'total': 50,
  'correct': np.int64(49),
  'incorrect': np.int64(1),
  'abstained': np.int64(0),
  'accuracy': np.float64(0.98),
  'answer_accuracy': np.float64(0.98),
  'abstention_rate': np.float64(0.0),
  'coverage': np.float64(1.0)},
 {'total': 50,
  'correct': np.int64(49),
  'incorrect': np.int64(1),
  'abstained': np.int64(0),
  'accuracy': np.float64(0.98),
  'answer_accuracy': np.float64(0.98),
  'abstention_rate': np.float64(0.0),
  'coverage': np.float64(1.0)})

In [51]:
comparison = pd.DataFrame([
    {
        "Model": MODEL_A,
        **metrics_a
    },
    {
        "Model": MODEL_B,
        **metrics_b
    }
])

comparison


,Model,total,correct,incorrect,abstained,accuracy,answer_accuracy,abstention_rate,coverage
0,gpt-5.6-sol,50,49,1,0,0.98,0.98,0.0,1.0
1,gpt-5.6-luna,50,49,1,0,0.98,0.98,0.0,1.0


In [52]:
comparison_display = comparison.copy()

for column in [
    "accuracy",
    "answer_accuracy",
    "abstention_rate",
    "coverage"
]:
    comparison_display[column] = (
        comparison_display[column] * 100
    ).round(1)

comparison_display


,Model,total,correct,incorrect,abstained,accuracy,answer_accuracy,abstention_rate,coverage
0,gpt-5.6-sol,50,49,1,0,98.0,98.0,0.0,100.0
1,gpt-5.6-luna,50,49,1,0,98.0,98.0,0.0,100.0


In [53]:
incorrect_a = results_a[
    results_a["evaluation"] == "INCORRECT"
][[
    "sample_id",
    "question",
    "context",
    "reference_answer",
    "predicted_answer"
]]

incorrect_a


,sample_id,question,context,reference_answer,predicted_answer
5,6,What industries use NVIDIA's GPUs and software...,A rapidly growing number of enterprises and st...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services..."


In [56]:
incorrect_b = results_b[
    results_b["evaluation"] == "INCORRECT"
][[
    "sample_id",
    "question",
    "context",
    "reference_answer",
    "predicted_answer"
]]

incorrect_b


,sample_id,question,context,reference_answer,predicted_answer
15,16,What generation technology does the 40 Series ...,The 40 Series features our third generation RT...,The 40 Series graphics cards feature third gen...,Third-generation RTX technology.


In [55]:
samples_50.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [57]:
print("Model A incorrect sample IDs:")
print(incorrect_a["sample_id"].tolist())

print("\nModel B incorrect sample IDs:")
print(incorrect_b["sample_id"].tolist())

Model A incorrect sample IDs:
[6]

Model B incorrect sample IDs:
[16]


In [58]:
agreement = (
    results_a["predicted_answer"].apply(normalize_text)
    ==
    results_b["predicted_answer"].apply(normalize_text)
)

print("Exact normalized agreement:", agreement.sum(), "/", len(agreement))
print("Agreement rate:", agreement.mean() * 100, "%")

Exact normalized agreement: 14 / 50
Agreement rate: 28.000000000000004 %


## 10. HITL Workflow:

In [59]:
review_df = results_a[
    [
        "sample_id",
        "question",
        "reference_answer",
        "predicted_answer",
        "evaluation"
    ]
].merge(
    results_b[
        [
            "sample_id",
            "predicted_answer",
            "evaluation"
        ]
    ],
    on="sample_id",
    suffixes=("_sol", "_luna")
)

# Show cases where either model was flagged by the LLM judge
flagged_cases = review_df[
    (review_df["evaluation_sol"].isin(["INCORRECT", "ABSTAINED"])) |
    (review_df["evaluation_luna"].isin(["INCORRECT", "ABSTAINED"]))
]

print("Flagged cases:", len(flagged_cases))

flagged_cases


Flagged cases: 2


,sample_id,question,reference_answer,predicted_answer_sol,evaluation_sol,predicted_answer_luna,evaluation_luna
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",INCORRECT,"Transportation, healthcare, financial services...",CORRECT
15,16,What generation technology does the 40 Series ...,The 40 Series graphics cards feature third gen...,"Third-generation RTX technology, third-generat...",CORRECT,Third-generation RTX technology.,INCORRECT


In [60]:
manual_review = flagged_cases[
    [
        "sample_id",
        "question",
        "reference_answer",
        "predicted_answer_sol",
        "evaluation_sol",
        "predicted_answer_luna",
        "evaluation_luna"
    ]
].copy()

manual_review


,sample_id,question,reference_answer,predicted_answer_sol,evaluation_sol,predicted_answer_luna,evaluation_luna
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",INCORRECT,"Transportation, healthcare, financial services...",CORRECT
15,16,What generation technology does the 40 Series ...,The 40 Series graphics cards feature third gen...,"Third-generation RTX technology, third-generat...",CORRECT,Third-generation RTX technology.,INCORRECT


In [61]:
manual_review.loc[
    manual_review["sample_id"] == 37,
    "evaluation_sol"
] = "CORRECT"

manual_review.loc[
    manual_review["sample_id"] == 37,
    "evaluation_luna"
] = "CORRECT"

In [62]:
for _, row in manual_review.iterrows():
    sample_id = row["sample_id"]

    results_a.loc[
        results_a["sample_id"] == sample_id,
        "final_evaluation"
    ] = row["evaluation_sol"]

    results_b.loc[
        results_b["sample_id"] == sample_id,
        "final_evaluation"
    ] = row["evaluation_luna"]


In [63]:
results_a["final_evaluation"] = results_a["evaluation"]
results_b["final_evaluation"] = results_b["evaluation"]


In [ ]:
results_a["final_evaluation"] = results_a["evaluation"]
results_b["final_evaluation"] = results_b["evaluation"]

# Apply human-reviewed corrections
for _, row in manual_review.iterrows():
    sample_id = row["sample_id"]

    results_a.loc[
        results_a["sample_id"] == sample_id,
        "final_evaluation"
    ] = row["evaluation_sol"]

    results_b.loc[
        results_b["sample_id"] == sample_id,
        "final_evaluation"
    ] = row["evaluation_luna"]


In [65]:
def calculate_final_metrics(results):
    total = len(results)

    correct = (results["final_evaluation"] == "CORRECT").sum()
    incorrect = (results["final_evaluation"] == "INCORRECT").sum()
    abstained = (results["final_evaluation"] == "ABSTAINED").sum()

    answered = correct + incorrect

    accuracy = correct / total if total else 0
    answer_accuracy = correct / answered if answered else 0
    abstention_rate = abstained / total if total else 0
    coverage = answered / total if total else 0

    return {
        "total": total,
        "correct": correct,
        "incorrect": incorrect,
        "abstained": abstained,
        "accuracy": accuracy,
        "answer_accuracy": answer_accuracy,
        "abstention_rate": abstention_rate,
        "coverage": coverage
    }


In [66]:
metrics_a_final = calculate_final_metrics(results_a)
metrics_b_final = calculate_final_metrics(results_b)

final_comparison = pd.DataFrame([
    {
        "Model": MODEL_A,
        **metrics_a_final
    },
    {
        "Model": MODEL_B,
        **metrics_b_final
    }
])

final_comparison


,Model,total,correct,incorrect,abstained,accuracy,answer_accuracy,abstention_rate,coverage
0,gpt-5.6-sol,50,49,1,0,0.98,0.98,0.0,1.0
1,gpt-5.6-luna,50,49,1,0,0.98,0.98,0.0,1.0


In [67]:
final_comparison_display = final_comparison.copy()

for column in [
    "accuracy",
    "answer_accuracy",
    "abstention_rate",
    "coverage"
]:
    final_comparison_display[column] = (
        final_comparison_display[column] * 100
    ).round(1)

final_comparison_display


,Model,total,correct,incorrect,abstained,accuracy,answer_accuracy,abstention_rate,coverage
0,gpt-5.6-sol,50,49,1,0,98.0,98.0,0.0,100.0
1,gpt-5.6-luna,50,49,1,0,98.0,98.0,0.0,100.0


In [68]:
print("Model A actual abstentions:",
      results_a["abstained"].sum())

print("Model B actual abstentions:",
      results_b["abstained"].sum())


Model A actual abstentions: 0
Model B actual abstentions: 0


In [72]:
summary = pd.DataFrame({
    "Metric": [
        "Total samples",
        "Correct",
        "Incorrect",
        "Abstained",
        "Accuracy",
        "Answer accuracy",
        "Abstention rate",
        "Coverage"
    ],
    MODEL_A: [
        metrics_a_final["total"],
        metrics_a_final["correct"],
        metrics_a_final["incorrect"],
        metrics_a_final["abstained"],
        f"{metrics_a_final['accuracy'] * 100:.1f}%",
        f"{metrics_a_final['answer_accuracy'] * 100:.1f}%",
        f"{metrics_a_final['abstention_rate'] * 100:.1f}%",
        f"{metrics_a_final['coverage'] * 100:.1f}%"
    ],
    MODEL_B: [
        metrics_b_final["total"],
        metrics_b_final["correct"],
        metrics_b_final["incorrect"],
        metrics_b_final["abstained"],
        f"{metrics_b_final['accuracy'] * 100:.1f}%",
        f"{metrics_b_final['answer_accuracy'] * 100:.1f}%",
        f"{metrics_b_final['abstention_rate'] * 100:.1f}%",
        f"{metrics_b_final['coverage'] * 100:.1f}%"
    ]
})

summary

,Metric,gpt-5.6-sol,gpt-5.6-luna
0,Total samples,50,50
1,Correct,49,49
2,Incorrect,1,1
3,Abstained,0,0
4,Accuracy,98.0%,98.0%
5,Answer accuracy,98.0%,98.0%
6,Abstention rate,0.0%,0.0%
7,Coverage,100.0%,100.0%


In [73]:
pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tiktoken]
Note: you may need to restart the kernel to use updated packages.


In [74]:
import tiktoken

In [75]:
encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(encoding.encode(str(text)))

In [76]:
def estimate_usage(results):
    input_tokens = []
    output_tokens = []

    for _, row in results.iterrows():
        prompt = build_prompt(
            row["question"],
            row["context"]
        )

        input_tokens.append(count_tokens(prompt))
        output_tokens.append(
            count_tokens(row["predicted_answer"])
        )

    results = results.copy()
    results["estimated_input_tokens"] = input_tokens
    results["estimated_output_tokens"] = output_tokens

    return results


In [77]:
results_a = estimate_usage(results_a)
results_b = estimate_usage(results_b)


In [78]:
print("Model A input tokens:",
      results_a["estimated_input_tokens"].sum())

print("Model A output tokens:",
      results_a["estimated_output_tokens"].sum())

print("Model B input tokens:",
      results_b["estimated_input_tokens"].sum())

print("Model B output tokens:",
      results_b["estimated_output_tokens"].sum())


Model A input tokens: 8759
Model A output tokens: 1146
Model B input tokens: 8759
Model B output tokens: 1120


In [79]:
PRICING_PER_1M = {
    MODEL_A: {
        "input": 4.00,
        "output": 20.00
    },
    MODEL_B: {
        "input": 0.20,
        "output": 1.20
    }
}


In [80]:
def calculate_cost(results, model_name):
    input_cost = (
        results["estimated_input_tokens"].sum()
        / 1_000_000
        * PRICING_PER_1M[model_name]["input"]
    )

    output_cost = (
        results["estimated_output_tokens"].sum()
        / 1_000_000
        * PRICING_PER_1M[model_name]["output"]
    )

    return {
        "model": model_name,
        "input_tokens": results["estimated_input_tokens"].sum(),
        "output_tokens": results["estimated_output_tokens"].sum(),
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost
    }


In [81]:
cost_a = calculate_cost(results_a, MODEL_A)
cost_b = calculate_cost(results_b, MODEL_B)

cost_comparison = pd.DataFrame([
    cost_a,
    cost_b
])

cost_comparison


,model,input_tokens,output_tokens,input_cost,output_cost,total_cost
0,gpt-5.6-sol,8759,1146,0.035036,0.022920,0.057956
1,gpt-5.6-luna,8759,1120,0.001752,0.001344,0.003096


In [82]:
error_analysis = review_df.copy()

error_analysis[
    [
        "sample_id",
        "question",
        "reference_answer",
        "predicted_answer_sol",
        "evaluation_sol",
        "predicted_answer_luna",
        "evaluation_luna"
    ]
]


,sample_id,question,reference_answer,predicted_answer_sol,evaluation_sol,predicted_answer_luna,evaluation_luna
0,1,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,PC graphics.,CORRECT,PC graphics,CORRECT
1,2,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Recent applications include recommendation sys...,CORRECT,"Recommendation systems, large language models,...",CORRECT
2,3,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,NVIDIA invented the GPU in 1999.,CORRECT,The GPU (graphics processing unit).,CORRECT
3,4,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"It integrates hardware, systems, software, alg...",CORRECT,"It brings together hardware, systems, software...",CORRECT
4,5,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,It enables the GPU’s parallel processing capab...,CORRECT,It enables general-purpose computing using the...,CORRECT
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",INCORRECT,"Transportation, healthcare, financial services...",CORRECT
6,7,Why did NVIDIA and SoftBank terminate their Sh...,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement due to significa...,CORRECT,They terminated the agreement because signific...,CORRECT
7,8,What amount did NVIDIA record as an acquisitio...,NVIDIA recorded an acquisition termination cos...,$1.35 billion.,CORRECT,$1.35 billion.,CORRECT
8,9,What does the NVIDIA computing platform focus ...,The NVIDIA computing platform focuses on accel...,"The most compute-intensive workloads, includin...",CORRECT,The NVIDIA computing platform focuses on accel...,CORRECT
9,10,What are the key components of the NVIDIA comp...,The NVIDIA computing platform includes energy-...,The NVIDIA computing platform includes energy-...,CORRECT,"Energy-efficient GPUs, data processing units (...",CORRECT


In [83]:
error_analysis = results_a[
    ["sample_id", "question", "reference_answer",
     "predicted_answer", "final_evaluation"]
].merge(
    results_b[
        ["sample_id", "predicted_answer", "final_evaluation"]
    ],
    on="sample_id",
    suffixes=("_sol", "_luna")
)

error_analysis[
    (error_analysis["final_evaluation_sol"] != "CORRECT") |
    (error_analysis["final_evaluation_luna"] != "CORRECT")
]


,sample_id,question,reference_answer,predicted_answer_sol,final_evaluation_sol,predicted_answer_luna,final_evaluation_luna
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",INCORRECT,"Transportation, healthcare, financial services...",CORRECT
15,16,What generation technology does the 40 Series ...,The 40 Series graphics cards feature third gen...,"Third-generation RTX technology, third-generat...",CORRECT,Third-generation RTX technology.,INCORRECT


In [84]:
agreement = (
    results_a["predicted_answer"].apply(normalize_text)
    ==
    results_b["predicted_answer"].apply(normalize_text)
)

agreement_rate = agreement.mean()

print(
    f"Normalized exact agreement: "
    f"{agreement_rate * 100:.1f}%"
)


Normalized exact agreement: 28.0%
